# Блокнот 9. Градиентный бустинг

Курс «Машинное обучение с нуля», модуль 9. Разделы: 1 — урок 9.1, 2 — урок 9.2, 3 — уроки 9.3–9.5, 4 — урок 9.4. Выполняйте ячейки сверху вниз; там, где `...  # ваш код`, допишите решение.

Первая ячейка ставит XGBoost, LightGBM и CatBoost, если их нет.

In [ ]:
%pip install -q xgboost lightgbm catboost

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import fetch_california_housing, make_moons
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

## 1. AdaBoost своими руками (урок 9.1)

**Задание 1.** Один шаг AdaBoost: по весам `w`, меткам `y` и ответам `h` (из {−1, +1}) верните голос `alpha` и новые нормированные веса.

In [ ]:
def adaboost_update(w, y, h):
    return ..., ...  # ваш код

In [ ]:
Xm, ym = make_moons(600, noise=0.3, random_state=0)
ym = 2 * ym - 1                                   # метки −1 и +1
Xtr, Xva, ytr, yva = train_test_split(Xm, ym, test_size=0.5, random_state=0)
w = np.full(len(ytr), 1 / len(ytr))
stumps, alphas = [], []
for t in range(100):
    stump = DecisionTreeClassifier(max_depth=1).fit(Xtr, ytr, sample_weight=w)
    alpha, w = adaboost_update(w, ytr, stump.predict(Xtr))
    stumps.append(stump)
    alphas.append(alpha)
F = sum(a * s.predict(Xva) for a, s in zip(alphas, stumps))
mine = np.mean(np.sign(F) == yva)
sk = AdaBoostClassifier(DecisionTreeClassifier(max_depth=1), n_estimators=100, random_state=0).fit(Xtr, ytr).score(Xva, yva)
print(f"Свой AdaBoost: {mine:.3f}; scikit-learn: {sk:.3f}")
assert mine > 0.85

## 2. Градиентный бустинг своими руками (урок 9.2)

**Задание 2.** Допишите цикл: остатки → дерево глубины 2 на остатках → шаг с learning rate.

In [ ]:
rng = np.random.default_rng(0)
x = np.sort(rng.uniform(0, 6, 200))
y = np.sin(x) * 2 + 0.3 * x + rng.normal(0, 0.3, 200)
X = x[:, None]
lr, trees = 0.1, []
F = np.full_like(y, y.mean())
for t in range(200):
    ...  # ваш код: остатки, дерево, шаг F
    trees.append(tree)
mse_mine = np.mean((y - F) ** 2)
sk = GradientBoostingRegressor(n_estimators=200, learning_rate=0.1, max_depth=2).fit(X, y)
print(f"MSE на обучении: свой бустинг {mse_mine:.4f}, scikit-learn {np.mean((y - sk.predict(X)) ** 2):.4f}")
assert abs(mse_mine - np.mean((y - sk.predict(X)) ** 2)) < 0.01
plt.scatter(x, y, s=8, color="gray")
plt.plot(x, F, lw=2)
plt.title("Градиентный бустинг из 200 деревьев глубины 2")
plt.show()

## 3. XGBoost, LightGBM и CatBoost на домах Калифорнии (уроки 9.3–9.5)

Разбиение то же, что в уроке 3.8. От обучающей части отделяем валидацию для ранней остановки.

In [ ]:
import catboost as cb
import lightgbm as lgb
import xgboost as xgb

Xc, yc = fetch_california_housing(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(Xc, yc, test_size=0.2, random_state=42)
X_fit, X_val, y_fit, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=0)
models = {
    "XGBoost": (xgb.XGBRegressor(n_estimators=5000, learning_rate=0.05, early_stopping_rounds=100, n_jobs=-1),
                {"eval_set": [(X_val, y_val)], "verbose": False}),
    "LightGBM": (lgb.LGBMRegressor(n_estimators=5000, learning_rate=0.05, verbose=-1),
                 {"eval_set": [(X_val, y_val)], "callbacks": [lgb.early_stopping(100, verbose=False)]}),
    "CatBoost": (cb.CatBoostRegressor(iterations=5000, learning_rate=0.05, early_stopping_rounds=100, verbose=0),
                 {"eval_set": (X_val, y_val)}),
}
results = {}
for name, (model, fit_kw) in models.items():
    t = time.perf_counter()
    model.fit(X_fit, y_fit, **fit_kw)
    p = model.predict(X_test)
    results[name] = r2_score(y_test, p)
    print(f"{name:9s} R² {results[name]:.3f}  RMSE {mean_squared_error(y_test, p) ** 0.5:.3f}  {time.perf_counter() - t:.1f} с")
assert min(results.values()) > 0.8

**Задание 3.** Обучите LightGBM с `learning_rate=0.3` и ранней остановкой. Сколько деревьев понадобилось и как изменилось качество?

In [ ]:
fast = ...  # ваш код
print("Деревьев:", fast.best_iteration_, " R² теста:", round(r2_score(y_test, fast.predict(X_test)), 3))

## 4. Категории без утечки (урок 9.4)

Условные данные: 1000 «планет» (у многих всего 1–3 объекта), у каждой свой эффект на цену доставки. Сравним наивное кодирование средним по всем обучающим данным и CatBoost с `cat_features`.

In [ ]:
rng = np.random.default_rng(1)
planets = rng.integers(0, 1000, 3000)
effect = rng.normal(0, 1, 1000)
distance = rng.uniform(0, 10, 3000)
price = effect[planets] + 0.3 * distance + rng.normal(0, 1, 3000)
tr = np.arange(3000) < 2000
te = ~tr

# наивно: среднее цены по планете, посчитанное по обучающим данным вместе с самим объектом
means = {p: price[tr & (planets == p)].mean() for p in np.unique(planets[tr])}
enc = np.array([means.get(p, price[tr].mean()) for p in planets])
naive = lgb.LGBMRegressor(verbose=-1).fit(np.c_[enc, distance][tr], price[tr])
print("Наивное кодирование: R² обучение", round(naive.score(np.c_[enc, distance][tr], price[tr]), 3),
      "тест", round(naive.score(np.c_[enc, distance][te], price[te]), 3))

Xcat = np.c_[planets.astype(str), distance].astype(object)
Xcat[:, 1] = distance
cat = cb.CatBoostRegressor(iterations=500, verbose=0, random_seed=0).fit(Xcat[tr], price[tr], cat_features=[0])
print("CatBoost с cat_features: R² обучение", round(cat.score(Xcat[tr], price[tr]), 3),
      "тест", round(cat.score(Xcat[te], price[te]), 3))